# Importing a public dataset : feeding state and larval locomotion

**Dr. Panagiotis Sakagiannis, Dr. Alexandros Marantis**

## About this notebook

An entry point to **larvaworld** built around a published dataset of freely crawling
*Drosophila* larvae in three metabolic states.

**The dataset.** Recordings from Jovanic et al. (2025), openly available on Zenodo. Each
recording is five minutes of undisturbed locomotion in a square arena, with no stimulus of
any kind, tracked as an eleven-point midline per animal.

**The biological question.** The dataset contains **three distinct larva groups**, one per
diet :

| group | diet | metabolic state |
|---|---|---|
| **Fed** | normal food | fed |
| **Sucrose** | sucrose only | protein-deprived |
| **Starved** | nothing | starved |

The distinct metabolic states might have an impact on locomotion. We focus on the
**temporal evolution of their dispersal in space** : how far the larvae of each group get
away from where they started, and how that distance grows over time.

This notebook is one of a series; a blank version is available as
`import_public_dataset_template.ipynb`.

## Setup

Importing larvaworld initializes its configuration registry : some components are loaded
from disc and the rest are built on the fly. `VERBOSE = 1` makes the import report what it
is doing, which is worth watching the first time.

In [ ]:
%matplotlib inline

from pathlib import Path

from IPython.display import display

import larvaworld
from larvaworld.lib import reg, sim, util
from larvaworld.lib.reg.generators import ReplayConf

larvaworld.VERBOSE = 1

# The name of this experiment. It labels the imported datasets and the output folders.
EXPERIMENT_NAME = "FeedingState"

MEDIA_DIR = Path(f"./media/{EXPERIMENT_NAME}")
plot_dir = (MEDIA_DIR / "plots").as_posix()
video_dir = (MEDIA_DIR / "videos").as_posix()

# Rendering the replay videos needs ffmpeg and takes several minutes.
MAKE_VIDEOS = False

ds = []  # the imported datasets, filled in further below

import shutil
import zipfile

import requests


def fetch_dataset(expect, archive, url=None, extract=True):
    """Make a dataset available locally, doing as little work as possible.

    Resolves in three steps, reporting which one it took :
      1. the extracted data is already there  -> nothing happens
      2. the archive is there but not unpacked -> unpack only
      3. neither                               -> download, then unpack

    Args:
        expect: path that exists once the data is unpacked.
        archive: path of the downloaded archive.
        url: where to download the archive from, if it is missing.
        extract: whether the archive can be unpacked here. RAR archives cannot.

    Returns:
        True if `expect` is available afterwards.
    """
    expect, archive = Path(expect), Path(archive)
    if expect.exists():
        print(f"[1/3] already present, nothing to do : {expect}")
        return True

    if not archive.exists():
        if url is None:
            print(f"[3/3] missing and no download link given : {archive}")
            return False
        archive.parent.mkdir(parents=True, exist_ok=True)
        print(f"[3/3] downloading {url}\n      -> {archive}")
        tmp = archive.with_suffix(archive.suffix + ".part")
        with requests.get(url, stream=True, timeout=60) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            done = 0
            with open(tmp, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
                    done += len(chunk)
                    if total:
                        print(
                            f"      {done / 1e6:8.0f} / {total / 1e6:.0f} MB", end="\r"
                        )
        tmp.rename(archive)
        print(f"\n      downloaded {archive.stat().st_size / 1e6:.0f} MB")
    else:
        print(f"[2/3] archive already downloaded : {archive}")

    if not extract:
        print(f"      this archive cannot be unpacked from Python. Extract it with")
        print(f"      7-Zip, WinRAR or unrar so that this exists :\n      {expect}")
        return expect.exists()

    print(f"      unpacking -> {expect.parent}")
    with zipfile.ZipFile(archive) as z:
        z.extractall(expect.parent)
    return expect.exists()

## Section 1 : Get the data

The dataset is openly available on Zenodo.

- **Record** : <https://zenodo.org/records/15075754>
- **Archive** : [`Main_Locomotion.rar`](https://zenodo.org/records/15075754/files/Main_Locomotion.rar?download=1)

The cell below fetches it, and skips the download if it is already on disc. It is a
**2.6 GB** file.

**One step here is not automatic.** Zenodo publishes this record as `.rar`, and Python
cannot unpack RAR archives - there is no support for it in the standard library, and the
tools that can (`unrar`, 7-Zip, WinRAR) are separate programs. The cell downloads the
archive and then tells you what to extract it into. Every other notebook in this series is
fully automatic; this one asks for one manual step because of how the data was published.

In [ ]:
DOWNLOAD_ROOT = Path.home() / "Downloads" / "Main_Locomotion"
URL = "https://zenodo.org/records/15075754/files/Main_Locomotion.rar?download=1"

fetch_dataset(
    expect=DOWNLOAD_ROOT,
    archive=DOWNLOAD_ROOT.with_suffix(".rar"),
    url=URL,
    extract=False,  # RAR cannot be unpacked from Python
)

RAW_FOLDER = (DOWNLOAD_ROOT / "Locomotion" / "Figure1").as_posix()
EXPERIMENT = "1a-d"

palette = {"Fed": "black", "Sucrose": "red", "Starved": "purple"}
gIDs = list(palette)


# The recording folders of a group : subfolders that actually hold data. Hidden and
# metadata folders that archives and network drives leave behind (".DS_Store",
# "@eaDir", ...) are skipped, as are empty folders.
def recording_folders(group_dir):
    return sorted(
        p.name
        for p in Path(group_dir).iterdir()
        if p.is_dir() and not p.name.startswith((".", "@")) and any(p.rglob("*"))
    )


DATA_AVAILABLE = all((Path(RAW_FOLDER) / EXPERIMENT / g).is_dir() for g in gIDs)
if DATA_AVAILABLE:
    for g in gIDs:
        print(
            f"{g:9s} : {len(recording_folders(Path(RAW_FOLDER) / EXPERIMENT / g))} recordings"
        )

## Section 2 : Import to larvaworld

Three things have to be specified : **where the data is**, **which tracker wrote it**, and
**which tracks to keep**.

### What the tracker recorded, and what it did not

Before importing, it is worth knowing which properties of a recording are written down
somewhere and which are not. For most published tracking data the picture is this :

| property | stated in the archive? | larvaworld can derive it |
|---|---|---|
| recording duration | usually, in the tracker's settings | not needed |
| stimulus protocol | usually, in the tracker's settings | not needed |
| **frame rate** | **often not** | **yes**, from the timestamps |
| **number of midline points** | **no** | **yes**, from the coordinates |
| **arena dimensions** | **no** | partly, see below |
| **pixel-to-millimetre scale** | **no** | no - you have to know it |

The highlighted rows are the ones that matter for the import, and they are the ones least
likely to be recorded. larvaworld therefore derives what it can from the data itself :

- **Frame rate.** Many trackers record at a variable rate, so a single nominal frame rate
  does not describe them. When a lab format declares a variable framerate, the timestep is
  measured from the timestamps and used for the whole import, including the stored dataset.
  Pass `estimate_dt=False` to keep the nominal value. Formats whose data carries no
  timestamps at all cannot use this, and need the frame rate set by hand.
- **Midline points.** Counted from the data and used whenever it disagrees with the
  expected number. Pass `estimate_midline_points=False` to switch this off.
- **Arena dimensions.** Estimated from the area the animals actually covered, which makes
  it a *lower bound* : larvae that never reach the rim make the arena look smaller than it
  is. It is therefore **off by default**. Pass `estimate_arena_dimensions=True`.

**The spatial scale is the one thing you must bring yourself.** If a tracker exports pixels
rather than millimetres, nothing in the coordinates reveals the conversion factor. A quick
sanity check settles which case you are in : the summed length of a larva's midline should
be a few millimetres for a third-instar larva. larvaworld applies that same check on
import and refuses data implying an impossible animal.

### Which tracker wrote it

This data comes from the Jovanic lab. That tracker records at a **variable** frame rate, so
its timestep is measured from the timestamps rather than assumed - watch the log below.

In [ ]:
lf = reg.conf.LabFormat.get("Jovanic")

### Which tracks to keep, and what to compute

Not every detected track is usable : the tracker loses and re-acquires animals, and short
fragments carry no information about dispersal.

In [ ]:
constraints = util.AttrDict(
    {
        "match_ids": False,  # use the tracker's own identities
        "interpolate_ticks": True,  # resample onto a regular time grid
        "min_duration_in_sec": 20,
        "time_slice": (0, 60),  # the first minute of each recording
    }
)

enr_kws = util.AttrDict(
    {
        "proc_keys": ["angular", "spatial"],
        "anot_keys": ["bout_detection"],
        "traj2origin": True,
        "tor_durs": [20],
        "dsp_starts": [0],
        "dsp_stops": [40, 60],
    }
)

### Putting it together

One dataset per group. The three `estimate_*` arguments are spelled out even where they
match the defaults, because they are the answer to the metadata question above.

In [ ]:
refIDs = [f"{EXPERIMENT_NAME}.{g}" for g in gIDs]

kws = {
    "raw_folder": RAW_FOLDER,
    "parent_dir": EXPERIMENT,
    "source_ids": gIDs,
    "group_id": EXPERIMENT_NAME,
    "refIDs": refIDs,
    "colors": [palette[g] for g in gIDs],
    "save_dataset": True,
    "estimate_dt": True,  # this tracker's framerate varies
    "estimate_midline_points": True,  # the default
    "estimate_arena_dimensions": False,  # the real arena is known, so do not guess it
    "enrich_conf": enr_kws,
    **constraints,
}

**This takes a few minutes** - it reads over a million rows, resamples them, and computes
the full metric set for each group. You only ever need to run it once.

Watch the log : it reports the timestep measured from the data, about 0.089 s, rather than
the nominal value the lab format carries. The import yields roughly **143 / 144 / 138**
larvae for Fed / Sucrose / Starved.

In [ ]:
if DATA_AVAILABLE:
    ds = lf.import_datasets(**kws)

    for d in ds:
        print(
            f"{d.id:9s} : {d.config.N} larvae, dt={d.config.dt:.4f} s, "
            f"{d.config.Npoints} midline points"
        )

### Reloading in a later session

In [ ]:
if not ds:
    if all(refID in reg.conf.Ref.confIDs for refID in refIDs):
        ds = [reg.loadRef(id=refID, load=True) for refID in refIDs]
        print("Loaded :", [d.id for d in ds])
    else:
        print("These datasets have not been imported yet. Run Section 2 first.")

## Section 3 : Data analysis and plotting

larvaworld ships a library of plotting routines, each registered under a short name. You
pick one by name and hand it the datasets you want compared - the group colors and labels
are taken from the datasets themselves, so every figure is consistent.

In [ ]:
# The available plots, by their unique IDs
print(reg.graphs.ks)

In [ ]:
# Arguments shared by every plot below. Figures are also written to `plot_dir`.
plot_kws = {"datasets": ds, "save_to": plot_dir, "show": False, "subfolder": None}

### The trajectories

First, simply what the larvae did : their paths over the analysed window.

In [ ]:
if ds:
    display(reg.graphs.run("trajectories", **plot_kws))

The same trajectories, but each one translated so that it starts at the origin, and colored
by group. This removes the arbitrary starting position of each animal and makes the *shape
and extent* of the paths directly comparable.

In [ ]:
if ds:
    display(
        reg.graphs.run("trajectories", mode="origin", single_color=True, **plot_kws)
    )

### Endpoint metrics

A boxplot of endpoint metrics - one value per larva, summarising its whole track. Each plot
routine has a default selection, but you can always name the metrics you want by their
short keys, as done here.

In [ ]:
if ds:
    display(
        reg.graphs.run(
            "endpoint box",
            ks=[
                "l",
                "fsv",
                "sv_mu",
                "run_tr",
                "pau_tr",
                "tor20_mu",
                "dsp_0_40_fin",
                "b_mu",
                "bv_mu",
            ],
            **plot_kws,
        )
    )

And a composite figure summarising exploration behavior across the groups.

In [ ]:
if ds:
    display(reg.graphs.run("exploration summary", **plot_kws))

### Dispersal

Dispersal is the distance of a larva from where it started. We compare the three metabolic states on it
in three increasingly informative ways.

**1. As an endpoint statistic.** The mean, final and maximum dispersal reached during the
analysed window - one number per larva, summarised as a boxplot per group.

In [ ]:
if ds:
    display(
        reg.graphs.run(
            "endpoint box",
            ks=["dsp_0_60_mu", "dsp_0_60_fin", "dsp_0_60_max"],
            **plot_kws,
        )
    )

**2. As a timecourse.** Dispersal plotted against time, showing both the mean and the
variance of each group : not just how far the groups got, but how fast and how consistently.

In [ ]:
if ds:
    display(reg.graphs.run("dispersal", **plot_kws))

In [ ]:
if ds:
    display(reg.graphs.run("dispersal", range=(0, 60), **plot_kws))

**3. Alongside the paths that produced it.** The summary versions place the timecourse next
to the corresponding trajectories, which makes the link between curve and behavior
immediate.

In [ ]:
if ds:
    display(reg.graphs.run("dispersal summary", **plot_kws))

## Section 4 : Visualize the dataset

A *replay* is a simulation whose agents are driven by recorded data instead of a model. It
gives you the same visualization tools you would use on a simulation - here, the
trajectories of all larvae of a group, transposed to a common origin and drawn as
accumulating trails.

Rendering needs `ffmpeg` (installed with larvaworld via `imageio_ffmpeg`) and takes a few
minutes per group, so it is off by default. Set `MAKE_VIDEOS = True` in the Setup cell.

In [ ]:
def run_replay(d):
    """Render one dataset's tracks to a video file in `video_dir`."""
    screen_kws = {
        "vis_mode": "video",
        "show_display": False,
        "draw_contour": False,
        "draw_midline": False,
        "draw_centroid": False,
        "visible_trails": True,
        "save_video": True,
        "fps": 1,
        "video_file": d.id,
        "media_dir": video_dir,
    }
    replay_conf = ReplayConf(
        transposition="origin", time_range=(0, 60), track_point=d.c.point_idx
    ).nestedConf
    rep = sim.ReplayRun(
        dataset=d,
        parameters=replay_conf,
        id=f"{d.id}_replay",
        screen_kws=screen_kws,
    )
    return rep.run()

In [ ]:
if MAKE_VIDEOS and ds:
    for d in ds:
        run_replay(d)

Finally the videos are stacked side by side into a single one, giving a direct visual
comparison of the groups.

In [ ]:
if MAKE_VIDEOS and ds:
    from larvaworld.lib.util.combining import combine_videos

    combine_videos(file_dir=video_dir, save_as="combined.mp4")
    print(f"Written to {video_dir}/combined.mp4")

## A few words on the lab format

Every tracker writes its own files, so larvaworld reads each one through a named **lab
format**. A lab format knows how a lab's files are laid out and how their contents must be
preprocessed, which is why the import above needed nothing more than a folder and a name.

| lab format | suits data that looks like |
|---|---|
| `Jovanic` | one file per recorded quantity, all animals stacked together |
| `Schleyer` | one file per animal, plus per-dish metadata |
| `Berni`, `Arguello` | one file per animal, columns in a fixed order |
| `DeepLabCut` | DeepLabCut CSV/HDF5 exports, one file per video |

Two things follow from this :

- **If one of them matches your tracker**, this notebook works on your own data with only
  the first section changed.
- **If none does**, a new lab format can be described and registered, after which your data
  imports like any other.

A lab format carries nominal values for things like the frame rate and the arena, because
they are usually constant for a lab. They describe the lab, not any particular recording,
which is why the import prefers what it can measure in the data itself.

## References

> Jovanic, T. *et al.* Feeding-state dependent neuropeptidergic modulation of reciprocally
> interconnected inhibitory neurons biases sensorimotor decisions in *Drosophila*.
> *Nature Communications* (2025). <https://doi.org/10.1038/s41467-025-61805-y>

> Jovanic, T., & Manceau, D. (2025). *Feeding-state dependent neuropeptidergic modulation of
> reciprocally interconnected inhibitory neurons biases sensorimotor decisions in Drosophila*
> [Dataset]. Zenodo. <https://doi.org/10.5281/zenodo.15075754>

Please cite both if you use this data.